# QC for CONGRADS
Initial filtering for CONGRADS:
- Resting state dicom instance 2 not null

Imaging qc:
- outside 3 std deviations

Genomic qc:
- Genetic sex == reported sex
- No sex chromosome aneuploidy
- Kinship
- Ancestry
- Variant QC performed later
    - missingness
    - HWE

Outputs:
- subject list that passed QC
- covariates genetic analyses dataframe
- behavioural phenotypes dataframe


TODO:
- Finish genetic QC with subject list from Barbara
- Make maybe list
- Finish covariates
- Finish exports

Last edit: 2023-11-21, J.S. Amelink

## 0. Set up

In [1]:
import pandas as pd
import os
import glob
import scipy as sp
import numpy as np

workspace_path = "/data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/"
cfs_path = "/data/clusterfs/lag/projects/lg-ukbiobank/working_data/imaging_data/CONGRADS_rest/"

#number of standard deviations for imaging QC
no_std = 3

#columns to do exclusion on:
qc_cols = [
    #T1
    'Amount of warping applied to non-linearly align T1 brain image to standard-space | Instance 2',
    "Discrepancy between rfMRI brain image and T1 brain image | Instance 2" ,
    'Inverted signal-to-noise ratio in T1 | Instance 2',
    
    #Resting state
    'Inverted temporal signal-to-noise ratio in artefact-cleaned pre-processed rfMRI | Instance 2',
    'Mean rfMRI head motion averaged across space and time points | Instance 2',
    '90th percentile of absolute head motion from rfMRI | Instance 2'
 ]

In [5]:
#load data and adapt labels
#df = pd.read_csv(os.path.join(workspace_path, 'CONGRADS_FLICA_participant.tsv'), sep="\t", engine="pyarrow", dtype={"eid": np.int64})
df = pd.read_csv(os.path.join(workspace_path, '65k_v2_participant.tsv'), sep="\t", engine="pyarrow", dtype={"eid": np.int64})

rename_dict = {"eid" : 'Participant ID',
               "p22001" : "Genetic sex",
               "p21003_i2" : 'Age when attended assessment centre | Instance 2',
               "p25733_i2" : 'Amount of warping applied to non-linearly align T1 brain image to standard-space | Instance 2',
               "p25739_i2" : 'Discrepancy between rfMRI brain image and T1 brain image | Instance 2',
               "p25756_i2" : 'Scanner lateral (X) brain position | Instance 2',
               "p25757_i2" : 'Scanner transverse (Y) brain position | Instance 2',
               "p25758_i2" : 'Scanner longitudinal (Z) brain position | Instance 2',
               "p25744_i2" : 'Inverted temporal signal-to-noise ratio in artefact-cleaned pre-processed rfMRI | Instance 2',
               "p25741_i2" : 'Mean rfMRI head motion averaged across space and time points | Instance 2',
               "p25737_i2" : 'Discrepancy between dMRI brain image and T1 brain image | Instance 2',
               "p25739_i2" : 'Discrepancy between rfMRI brain image and T1 brain image | Instance 2',
               "p25922_i2" : 'Standard deviation of apparent translation in the Y axis as measured by eddy | Instance 2',
               "p25746_i2" : 'Number of dMRI outlier slices detected and corrected | Instance 2',
               "p22000" : 'Genotype measurement batch',
               "p32050" : 'Exome release tranche',
               "p22009_a1" :'Genetic principal components | Array 1',
               "p22009_a2" : 'Genetic principal components | Array 2',
               "p22009_a3" :'Genetic principal components | Array 3',
               "p22009_a4" : 'Genetic principal components | Array 4',
               "p22009_a5" : 'Genetic principal components | Array 5',
               "p22009_a6" : 'Genetic principal components | Array 6',
               "p22009_a7" : 'Genetic principal components | Array 7',
               "p22009_a8" : 'Genetic principal components | Array 8',
               "p22009_a9" : 'Genetic principal components | Array 9',
               "p22009_a10" : 'Genetic principal components | Array 10',
               "p54_i2" : "UK Biobank assessment centre | Instance 2",
               "p1707_i2" : 'Handedness (chirality/laterality) | Instance 2',
               "p1707_i0" : 'Handedness (chirality/laterality) | Instance 0',
               "p41270" : 'Diagnoses - ICD10',
               "p41204" : 'Diagnoses - secondary ICD10',
               "p20016_i2" : "Fluid intelligence score | Instance 2",
               "p22019" : "Sex chromosome aneuploidy",
               "p20252_i2" : "T1 structural brain images - NIFTI | Instance 2",
               "p20250_i2" : "Multiband diffusion brain images - NIFTI | Instance 2",
               "p20227_i2" : "Functional brain images - resting - NIFTI | Instance 2",
               "p25734_i2" : "Inverted signal-to-noise ratio in T1 | Instance 2",
               "p31" : "Sex",
                "p21022" : "Age at recruitment",
                "p26302_i2" : "Specific cognitive ability (AS) | Instance 2",
                "p6364_i2" : "Vocabulary level | Instance 2",
                "p6365_i2" : "Uncertainty in vocabulary level | Instance 2",
                "p24441_i2" : "Mean relative head motion from rfMRI  | Instance 2",
                "p24439_i2" : "Median absolute head motion from rfMRI  | Instance 2",
                "p24412_i2" : "T1 acquisition protocol versions | Instance 2",
                "p24442_i2" : "Median relative head motion from rfMRI | Instance 2",
                'p24438_i2': "Mean absolute head motion from rfMRI | Instance 2",
               'p24440_i2': "90th percentile of absolute head motion from rfMRI | Instance 2" ,
               'p24443_i2': "90th percentile of relative head motion from rfMRI | Instance 2"
                 }


#column_labels = pd.read_csv(os.path.join(workspace_path, "FLICA_table_v2_lables_participant.tsv"), sep="\t")
#rename_dict = dict(zip(list(df.columns), list(column_labels)))
df.rename(columns=rename_dict, inplace=True)

In [6]:
gwas_check_we_file = open("/data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/European_97perc_cluster_PC1cutoff.tsv")
gwas_check_list = [int(x) for x in sorted(list(gwas_check_we_file.read().split('\n')))[1:-1]]
gwas_check = np.bool_( [True if x in gwas_check_list else False for x in df["Participant ID"] ] )
df = df[gwas_check]
print(len(df))

56096


## 1. Imaging QC

In [3]:
#get participants with data on cluster
rs_data_zip = list(glob.glob("/data/workspaces/lag/workspaces/lg-ukbiobank/primary_data/imaging_data/*/*_20227_2_0.zip"))
rs_data_cluster = list(glob.glob("/data/clusterfs/lag/projects/lg-ukbiobank/primary_data/imaging_data/*/fMRI/rfMRI.ica/filtered_func_data_clean.nii.gz"))
#diff_data_zip = list(glob.glob("/data/workspaces/lag/workspaces/lg-ukbiobank/primary_data/imaging_data/*/*_20250_2_0.zip"))
rs_part_zip = [int(x[-21:-14]) for x in rs_data_zip]
rs_part_cluster = [int(x[-54:-47]) for x in rs_data_cluster]

#print("Number of subjects with diffusion data on the cluster: {}".format(len(diff_data_zip)))
print("Number of subjects with resting state data in the house: {}".format(len(rs_data_zip)))
print("Number of subjects with resting state data on the cluster: {}".format(len(rs_data_cluster)))

#overlap = []
rs_cluster_set = set(rs_part_cluster)
#for sub in rs_part_zip:
#    if sub in diff_part_set:
#        overlap.append(sub)
    
#print("Number of subjects with resting state data on the cluster: {}".format(len(overlap)))
imaging_cluster = np.bool_( [True if x in rs_cluster_set else False for x in df["Participant ID"] ] ).reshape((1, len(df)))

Number of subjects with resting state data in the house: 66754
Number of subjects with resting state data on the cluster: 60394


In [7]:
#check missing data
no_part_start = len(df)
print("Number of participants in dataframe: {}".format(no_part_start))

qc_missing_rs = np.bool_(df['Mean rfMRI head motion averaged across space and time points | Instance 2'].isna()).reshape((1, len(df)))
qc_missing_rs_2 = np.bool_(df['Scanner lateral (X) brain position | Instance 2'].isna()).reshape((1, len(df)))
#qc_missing_diff = np.bool_(df[ 'Number of dMRI outlier slices detected and corrected | Instance 2'].isna()).reshape((1, len(df)))
qc_missing = np.logical_or(qc_missing_rs, qc_missing_rs_2)

print("Subjects without resting state imaging qc metrics: {}".format(np.sum(qc_missing_rs)))
print("Subjects without resting state imaging qc metrics: {}".format(np.sum(qc_missing_rs_2)))
print("Subjects without resting state imaging both qc metrics: {}".format(np.sum(qc_missing)))
#print("Subjects without diffusion imaging qc metrics: {}".format(np.sum(qc_missing_diff)))
#print("Subjects without one or more imaging qc metrics: {}".format(np.sum(qc_missing)))

Number of participants in dataframe: 56096
Subjects without resting state imaging qc metrics: 3686
Subjects without resting state imaging qc metrics: 4045
Subjects without resting state imaging both qc metrics: 6468


In [10]:
#imaging qc:
stds=df[qc_cols].std()
means=df[qc_cols].mean()

high_cut_off = means+no_std*stds
high = np.bool_([np.array(df[qc_cols] < high_cut_off).sum(axis=1) < len(qc_cols)]).reshape((1, len(df)))

high_excl = set(df.loc[high.T, "Participant ID"]) - set(df.loc[qc_missing_rs.T, "Participant ID"])

#missing = np.logical_or(qc_missing, ~imaging_cluster)
imaging_qc_all = np.logical_or(high, qc_missing) #qc_missing_rs)

print("Subjects outside {0} standard deviations: {1}".format( no_std, len(high_excl)) )
print("Subjects that failed metric QC outside {0} standard deviations (includes missing QC metrics): {1}".format(no_std, np.sum(high)))
#print("Subjects with data + QC available: {}".format(np.sum(~missing)))
#print("Subjects missing: {}".format(np.sum(missing)))
print("Number of subjects excluded after imaging QC (or data unavailable on cluster): {}".format(np.sum(imaging_qc_all)))

df = df[~imaging_qc_all.T]

Subjects outside 3 standard deviations: 3397
Subjects that failed metric QC outside 3 standard deviations (includes missing QC metrics): 7083
Number of subjects excluded after imaging QC (or data unavailable on cluster): 9679


## 2. Exporting:
- include list
- exclude list
- missing list (maybe second round)
- filtered dataframe
- behavioural 
- covariates

In [14]:
#participant lists
include_list = df.loc[:, "Participant ID"].to_csv(os.path.join(workspace_path,"participant_list_imaging_gen_check_new_65k_N{}.txt".format(len(df))),index=False,header=False, sep="\t") 

#maybe_list = df.loc[maybe_bool.T, "Participant ID"].to_csv(os.path.join(workspace_path,"maybe_participants_post_gencheck_N{}.txt".format(np.sum(maybe_bool))),index=False,header=False, sep="\t")

#keep = df.loc[~basic_filter.T, :]
df.to_csv(os.path.join(workspace_path,"CONGRADS_filtered_data_new_65k_N{}.tsv".format(len(df))), index=False, sep="\t")

print("Number of participants included: {}".format(df.shape))

Number of participants included: (46417, 44)


In [13]:
cat /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participant_list_imaging_gen_check_N29612.txt | head -n 100 > /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participants_hundred.txt
cat /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participant_list_imaging_gen_check_N29612.txt | head -n 1000 | tail -n 900 > /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participants_thousand.txt 
cat /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participant_list_imaging_gen_check_N29612.txt | head -n 10000 | tail -n 9000 > /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participants_ten_thousand.txt 
cat /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participant_list_imaging_gen_check_N29612.txt | head -n 20000 | tail -n 10000 > /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participants_next_ten_thousand.txt 
cat /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participant_list_imaging_gen_check_N29612.txt | tail -n 9612 > /data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/participants_remainder.txt 

cat: write error: Broken pipe
cat: write error: Broken pipe


In [18]:
#covariates
covariates = [ #subject characteristics
              'Participant ID',
              'Genetic sex',
              'Age when attended assessment centre | Instance 2',
    
                #T1
              'Amount of warping applied to non-linearly align T1 brain image to standard-space | Instance 2',
              'Scanner lateral (X) brain position | Instance 2',
              'Scanner transverse (Y) brain position | Instance 2',
              'Scanner longitudinal (Z) brain position | Instance 2',
    
                #rfMRI
              'Inverted temporal signal-to-noise ratio in artefact-cleaned pre-processed rfMRI | Instance 2',
              'Mean rfMRI head motion averaged across space and time points | Instance 2',
              'Discrepancy between rfMRI brain image and T1 brain image | Instance 2',
              "90th percentile of absolute head motion from rfMRI | Instance 2", 
    
                #genetic
              'Genetic principal components | Array 1',
              'Genetic principal components | Array 2',
              'Genetic principal components | Array 3',
              'Genetic principal components | Array 4',
              'Genetic principal components | Array 5',
              'Genetic principal components | Array 6',
              'Genetic principal components | Array 7',
              'Genetic principal components | Array 8',
              'Genetic principal components | Array 9',
              'Genetic principal components | Array 10'
]

covs = df[covariates]

covs['age_sq'] = np.square(covs['Age when attended assessment centre | Instance 2'])
covs['age_sex'] = covs['Age when attended assessment centre | Instance 2']*covs['Genetic sex']

#make dummies
covs['geno_array_dummy'] = 1
covs.loc[df['Genotype measurement batch'] < 0, 'geno_array_dummy'] = 0

site_dummies = pd.get_dummies(df['UK Biobank assessment centre | Instance 2'])
site_dummies.columns = ["site_dummy_{0}".format(x) for x in site_dummies.columns]

#covs['exome_dummy'] = 1
#covs.loc[keep["Exome release tranche"] > 3.1, 'exome_dummy'] = 0

covs = pd.concat([covs, site_dummies], axis=1)
covs.columns = [x.replace(" ","_") for x in covs.columns]

covs = covs[ ['Participant_ID'] + list(covs.columns[0:]) ]

covs.columns = [ ['FID', 'IID'] + list(covs.columns[2:]) ]

covs.to_csv(os.path.join(workspace_path, "regenie_covariates_65k.tsv"), sep="\t", index=False)

/tmp/ipykernel_14860/2981812327.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  covs['age_sq'] = np.square(covs['Age when attended assessment centre | Instance 2'])
/tmp/ipykernel_14860/2981812327.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  covs['age_sex'] = covs['Age when attended assessment centre | Instance 2']*covs['Genetic sex']
/tmp/ipykernel_14860/2981812327.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_